# Flight Data Cleaning

Clean flight delay data, rename columns for clarity and convert from csv to dataframe.

---

## Table of Contents

1. [Setup](#1-setup)
2. [Load Data](#2-load-data)
3. [Examine Data Structure](#3-examine-data-structure)
4. [Calculate Target Variables](#4-calculate-target-variables)
5. [Data Quality Checks](#5-data-quality-checks)
6. [Rename Columns for Clarity](#6-rename-columns-for-clarity)
7. [Save Cleaned Data](#7-save-cleaned-data)
8. [Summary](#8-summary)

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import requests
import glob
from datetime import datetime
from dateutil.relativedelta import relativedelta

## 2. Load Data

Download the latest data from BITRE. The script will attempt to download the most recent dataset automatically.

**If automatic download fails**, manually download from:  
https://www.bitre.gov.au/publications/ongoing/airline_on_time_monthly

Save the file to `data/raw/` folder.

In [ ]:
def download_bitre_data(save_path="../data/raw/"):
    """
    Download the latest BITRE flight data by trying recent months.
    Tries the month before current date first, then the month before that.
    """
    base_url = "https://www.bitre.gov.au/sites/default/files/documents/"
    current_date = datetime.now()
    
    # Use browser-like headers to avoid being blocked
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
    }
    
    # Try the last 2 months (month before current, then month before that)
    for months_back in range(1, 3):
        target_date = current_date - relativedelta(months=months_back)
        month_name = target_date.strftime("%B").lower()
        year = target_date.year
        
        filename = f"OTP_Time_Series_Master_Current_{month_name}_{year}.xlsx"
        url = base_url + filename
        
        print(f"Attempting to download: {filename}")
        
        try:
            response = requests.get(url, headers=headers, timeout=60)
            if response.status_code == 200:
                filepath = save_path + filename
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                print(f"✓ Successfully downloaded: {filename}")
                return filepath
            else:
                print(f"  Not found (HTTP {response.status_code})")
        except requests.RequestException as e:
            print(f"  Failed: {type(e).__name__}")
    
    print("\n⚠ Automatic download failed. Will use existing local file if available.")
    return None

# Attempt to download latest BITRE data
downloaded_file = download_bitre_data()
print(f"\n{'='*80}")

In [ ]:
# Check if downloaded file is available, otherwise find the latest local file
if downloaded_file:
    data_file = downloaded_file
else:
    # Find the most recent local file
    local_files = glob.glob("../data/raw/OTP_Time_Series_Master_Current_*.xlsx")
    if local_files:
        data_file = max(local_files)  # Get the most recent by filename
        print(f"Using existing local file: {data_file}")
    else:
        raise FileNotFoundError("No BITRE data file found. Please download manually.")

# Load all sheets from the Excel file
all_sheets_temp = pd.read_excel(data_file, sheet_name=None)
df = pd.concat(all_sheets_temp.values(), ignore_index=True)

# Remove duplicates caused by overlapping data across sheets
# (e.g., '2020-25 OTP' sheet contains 2020 data that also exists in '2020' sheet)
rows_before = len(df)
df = df.drop_duplicates()
rows_dropped = rows_before - len(df)
if rows_dropped > 0:
    print(f"Removed {rows_dropped} duplicate rows from overlapping sheets")

# Convert date to datetime for easier sorting
df['Month'] = pd.to_datetime(df['Month'], format='%m/%Y')
df['year_month'] = df['Month'].dt.to_period('M')

print(f"Loaded data from: {data_file}")
print(f"{'='*80}")
print(f"Number of airlines: {df['Airline'].nunique()}")
print(f"Total records: {len(df)}")

Let's start with:
* one route pair between two cities: between Sydney and Melbourne.
* individual airline performance (excluding "All Airlines" aggregate)

In [ ]:
df_select = df[((df["Route"] == "Sydney-Melbourne") | (df["Route"] == "Melbourne-Sydney")) &
               (df["Airline"] != "All Airlines")].reset_index()

print(f"\n{'='*80}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")
print(f"Number of rows: {len(df_select)}")
print(f"Airlines: {df_select['Airline'].unique().tolist()}")

## 3. Examine Data Structure

The columns and sample data are examined below to understand the dataset structure.

In [ ]:
# Display column names
print("Column names:")
print(df_select.columns.tolist())
print(f"\n{"="*80}")

# Display basic info
print("\nDataset shape:", df_select.shape)
print(f"\n{"="*80}")

# Display first few rows
print("\nFirst 5 rows:")
df_select.head()

In [ ]:
# Check data types and missing values
print("Data types and missing values:")
print(df_select.info())
print(f"\n{"="*80}")

# Check for missing values
print("\nMissing values per column:")
print(df_select.isnull().sum())

## 4. Calculate Target Variables

Calculate the primary target variable (delay_rate) and secondary target variable (is_high_delay) as defined in the problem definition document.

In [ ]:
# Create a copy for cleaning
df_clean = df_select.copy()

# Check for any missing or zero values in Sectors Flown
print(f"Missing values in Sectors Flown: {df_clean['Sectors Flown'].isna().sum()}")
print(f"Zero values in Sectors Flown: {(df_clean['Sectors Flown'] == 0).sum()}")
print(f"\n{'='*80}")

# Filter out rows where Sectors Flown == 0 (prevents division by zero)
# These are typically months where airlines suspended operations (e.g., Rex Airlines Aug 2024, Tigerair Jul 2011)
rows_before = len(df_clean)
df_clean = df_clean[df_clean['Sectors Flown'] > 0].reset_index(drop=True)
rows_removed = rows_before - len(df_clean)
print(f"Removed {rows_removed} rows with Sectors Flown = 0")
print(f"Rows remaining: {len(df_clean)}")
print(f"\n{'='*80}")

# Calculate delay_rate (Primary target variable)
# delay_rate = (Sectors_Flown - Arrivals_On_Time) / Sectors_Flown
df_clean['delay_rate'] = (df_clean['Sectors Flown'] - df_clean['Arrivals On Time']) / df_clean['Sectors Flown']

# Calculate binary classification target (is_high_delay)
# Using 25% threshold as defined in problem definition
df_clean['is_high_delay'] = (df_clean['delay_rate'] > 0.25).astype(int)

print("\nTarget variables calculated:")
print(f"delay_rate - Mean: {df_clean['delay_rate'].mean():.4f}, Median: {df_clean['delay_rate'].median():.4f}")
print(f"is_high_delay - High delay months: {df_clean['is_high_delay'].sum()} out of {len(df_clean)} ({df_clean['is_high_delay'].mean()*100:.1f}%)")
print(f"\n{'='*80}")

# Verify no NaN or Inf values in delay_rate
print(f"\nVerifying delay_rate calculation:")
print(f"NaN values: {df_clean['delay_rate'].isna().sum()}")
print(f"Inf values: {(df_clean['delay_rate'] == float('inf')).sum()}")

## 5. Data Quality Checks

Before proceeding, we need to:
1. Check for missing values in critical columns
2. Identify months with very low flight counts
3. Exclude COVID period (April-December 2020) as per problem definition

In [ ]:
# 5.1 Check data distribution by year
print("Records per year:")
df_clean['year'] = df_clean['Month'].dt.year
print(df_clean['year'].value_counts().sort_index())
print(f"\n{"="*80}")

# 5.2 Identify COVID period
covid_mask = (df_clean['Month'] >= '2020-04-01') & (df_clean['Month'] <= '2020-12-31')
print(f"\nCOVID period rows (Apr-Dec 2020): {covid_mask.sum()}")
print(f"\n{"="*80}")

# 5.3 Show date range before filtering
print(f"\nDate range before filtering: {df_clean['year_month'].min()} to {df_clean['year_month'].max()}")
print(f"Total rows before filtering: {len(df_clean)}")

In [ ]:
# 5.4 Exclude COVID period (April-December 2020)
df_clean = df_clean[~covid_mask].reset_index(drop=True)

print(f"Date range after COVID exclusion: {df_clean['year_month'].min()} to {df_clean['year_month'].max()}")
print(f"Total rows after COVID exclusion: {len(df_clean)}")
print(f"\n{"="*80}")

# Verify no COVID period data remains
print("\nVerifying COVID period exclusion:")
print(f"Any rows in Apr-Dec 2020? {((df_clean['Month'] >= '2020-04-01') & (df_clean['Month'] <= '2020-12-31')).any()}")

## 6. Rename Columns for Clarity

Rename columns to follow consistent naming conventions (snake_case, clear names).

In [ ]:
# Rename columns to snake_case for consistency
column_mapping = {
    'Route': 'route',
    'Departing Port': 'departing_port',
    'Arriving Port': 'arriving_port',
    'Airline': 'airline',
    'Month': 'month',
    'Sectors Scheduled': 'sectors_scheduled',
    'Sectors Flown': 'sectors_flown',
    'Cancellations': 'cancellations',
    'Departures On Time': 'departures_on_time',
    'Arrivals On Time': 'arrivals_on_time',
    'Departures Delayed': 'departures_delayed',
    'Arrivals Delayed': 'arrivals_delayed',
    'OnTime Departures \n(%)': 'ontime_departures_pct',
    'OnTime Arrivals \n(%)': 'ontime_arrivals_pct',
    'Cancellations \n\n(%)': 'cancellations_pct',
    'year_month': 'year_month'
}

df_clean = df_clean.rename(columns=column_mapping)

print("Columns after renaming:")
print(df_clean.columns.tolist())
print(f"\n{'='*80}")

# Drop the 'index' column from the reset_index operation if it exists
if 'index' in df_clean.columns:
    df_clean = df_clean.drop(columns=['index'])
    print("Dropped 'index' column")
    
print(f"\nFinal shape: {df_clean.shape}")

## 7. Save Cleaned Data

Save the cleaned dataframe to the processed data folder.

In [ ]:
# Select final columns to keep
# Note: 'route' is excluded as it's redundant - 'departing_port' and 'arriving_port' 
# provide the same directional information in a more ML-friendly format
columns_to_keep = [
    'departing_port', 'arriving_port', 'airline', 
    'month', 'year_month', 'year',
    'sectors_scheduled', 'sectors_flown', 
    'arrivals_on_time', 'arrivals_delayed',
    'cancellations', 'cancellations_pct',
    'delay_rate', 'is_high_delay'
]

df_final = df_clean[columns_to_keep].copy()

print("Final dataset summary:")
print(f"Shape: {df_final.shape}")
print(f"Date range: {df_final['year_month'].min()} to {df_final['year_month'].max()}")
print(f"Departing ports: {df_final['departing_port'].unique()}")
print(f"Arriving ports: {df_final['arriving_port'].unique()}")
print(f"\n{'='*80}")

# Display final columns
print("\nFinal columns:")
print(df_final.columns.tolist())
print(f"\n{'='*80}")

# Save to processed data folder
output_path = "../data/processed/flight_data_cleaned_syd_mel.csv"
df_final.to_csv(output_path, index=False)
print(f"\n✓ Cleaned data saved to: {output_path}")

# Display sample of final data
print(f"\n{'='*80}")
print("\nSample of cleaned data:")
print(df_final.head(10))

## 8. Summary

**Data Cleaning Steps Completed:**

1. ✓ Loaded BITRE flight data from Excel (all sheets)
2. ✓ Filtered to Sydney-Melbourne route pairs (both directions)
3. ✓ Filtered to individual airlines (excluding "All Airlines" aggregate)
4. ✓ Removed rows with Sectors Flown = 0 (prevents division by zero)
5. ✓ Excluded COVID period (April-December 2020)
6. ✓ Calculated target variables:
   - `delay_rate`: Continuous variable (0-1) representing proportion of delayed arrivals
   - `is_high_delay`: Binary variable (1 if delay_rate > 0.25, else 0)
7. ✓ Renamed columns to snake_case for consistency
8. ✓ Selected relevant columns for analysis
9. ✓ Saved cleaned data to `data/processed/flight_data_cleaned_syd_mel.csv`

**Next Step:**
- Weather data cleaning and feature engineering (see notebook *1b_compute_weather_stats_feature_batch.ipynb*)
- Merge flight and weather data (see notebook *3_prepare_ml_training_data_single_route.ipynb*)
- Exploratory Data Analysis and model development